In [3]:
import numpy
print(numpy)
print(numpy.__file__)


<module 'numpy' (namespace) from ['/Users/anitaganesan/Documents/Movie_Project/.venv/lib/python3.12/site-packages/numpy']>
None


In [13]:
ratings = pd.read_csv(
    "ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    "ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None
)

movies = movies[[0, 1]]
movies.columns = ["movie_id", "title"]

In [14]:
ratings = ratings.iloc[:200000]

In [15]:
df = ratings.merge(movies, on="movie_id")

In [16]:
user_movie_matrix = df.pivot_table(
    index="user_id",
    columns="title",
    values="rating"
)

In [22]:
def recommend_movies(movie_title, top_n=10):
    # Check if movie exists in the dataset

    while movie_title not in user_movie_matrix.columns:
        print("Movie is not found in dataset.")
        movie_title = input ("Please enter another movie title: ")

    # Get ratings for the selected movie
    movie_ratings = user_movie_matrix[movie_title]

    # Calculate Pearson correlation between this movie
    # and every other movie
    similar = user_movie_matrix.corrwith(movie_ratings)

    # Convert correlations to a DataFrame
    corr_df = pd.DataFrame(similar, columns=["correlation"])

    # Remove invalid correlations (NaN)
    corr_df = corr_df.dropna()

    # Count how many ratings each movie has received
    rating_counts = df.groupby("title")["rating"].count()

    # Add rating counts to the correlation table
    corr_df["num_ratings"] = rating_counts

    # Remove movies with too few ratings
    corr_df = corr_df[corr_df["num_ratings"] >= 30]

    # Remove the original movie from recommendations
    corr_df = corr_df.drop(movie_title, errors="ignore")

    # Sort by correlation and return top results
    return corr_df.sort_values(
        "correlation",
        ascending=False
    ).head(top_n)

In [ ]:
recommend_movies("Obsession")

Movie is not found in dataset.
